# Step 4: Time-Aware Train / Validation / Test Split

## 1. Objective

Split the engineered feature dataset (`data/processed/paysim_features.parquet`) into train, validation, and test sets using a chronological rule -- not a random one -- and verify that the split respects real time. No model is trained in this notebook; all logic lives in `src/split_data.py`, demonstrated and validated here.

## 2. Why random splitting is inappropriate

A fraud model is deployed against transactions that haven't happened yet -- it is trained on the past and scored against the future. `train_test_split(shuffle=True)` or a random stratified split would let the model train on rows that occur, in simulated time, AFTER some of its own validation/test rows. That is an evaluation setup no real deployment could ever match, and it would make the reported metrics optimistic in a way that wouldn't survive contact with production. A chronological split -- train = earliest steps, validation = middle steps, test = latest steps -- is the only split that mirrors how the model will actually be used.

## 3. Load processed feature data

In [ ]:
import sys
sys.path.append("..")

import json
import os
import time
import numpy as np
import pandas as pd

from src.data_utils import load_raw_data
from src.features import build_feature_dataset, get_model_feature_columns
from src.split_data import (
    inspect_temporal_distribution,
    choose_temporal_boundaries,
    create_temporal_split,
    validate_temporal_split,
    summarize_split,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 40)

df = pd.read_parquet("../data/processed/paysim_features.parquet")
print("Loaded:", df.shape)

## 4. Inspect temporal distribution

Before picking any boundary, look at how transactions and fraud are actually distributed across the 743 simulated steps -- shown compactly at the simulated-day level (24 steps/day).

In [ ]:
print("min step:", df["step"].min(), " max step:", df["step"].max(), " unique steps:", df["step"].nunique())

per_step = inspect_temporal_distribution(df)
per_step.head(10)

In [ ]:
day = (per_step.index - 1) // 24
per_day = per_step.groupby(day).agg(txn_count=("txn_count", "sum"), fraud_count=("fraud_count", "sum"))
per_day["fraud_rate_pct"] = per_day["fraud_count"] / per_day["txn_count"] * 100
per_day["cum_txn_pct"] = per_day["txn_count"].cumsum() / len(df) * 100
per_day["cum_fraud_pct"] = per_day["fraud_count"].cumsum() / df["isFraud"].sum() * 100
per_day.index.name = "simulated_day"
per_day

**Finding:** transaction volume is heavily front-loaded and decays sharply after simulated day ~16 (day 16: 425,766 txns; day 21: 53,437; day 30: only 272), while fraud COUNT stays roughly steady throughout (~250-320/day). The combination means the fraud RATE rises sharply in the later days -- day 26 is 3.26%, day 30 is **100%** (272 transactions, all fraud, zero legitimate transactions that day). This is a genuine property of the raw dataset (confirmed in Step 2 EDA), not something introduced by splitting, and it directly shapes what the test period will look like.

## 5. Choose chronological boundaries

Boundaries are chosen as the earliest step at which cumulative **row count** reaches ~70% and ~85% of the full dataset -- always landing on a complete step (a step's transactions are never split across two sets).

In [ ]:
train_end_step, val_end_step = choose_temporal_boundaries(per_step, train_pct=0.70, val_pct=0.15)
max_step = int(df["step"].max())

print(f"TRAIN:      step 1 -> step {train_end_step}")
print(f"VALIDATION: step {train_end_step + 1} -> step {val_end_step}")
print(f"TEST:       step {val_end_step + 1} -> step {max_step}")
print(f"\nCumulative row %% at train boundary: {per_step.loc[train_end_step, 'cum_txn_pct']:.2f}%")
print(f"Cumulative row %% at validation boundary: {per_step.loc[val_end_step, 'cum_txn_pct']:.2f}%")

These land at 70.15% / (85.56% - 70.15% =) 15.41% / (100% - 85.56% =) 14.44% -- close to the 70/15/15 target without forcing an exact split, since the underlying daily volume is too uneven to hit exact thirds cleanly.

## 6. Create train/validation/test sets

In [ ]:
train_df, val_df, test_df = create_temporal_split(df, train_end_step, val_end_step)
print(f"train:      {train_df.shape}")
print(f"validation: {val_df.shape}")
print(f"test:       {test_df.shape}")

## 7 & 8. Validate temporal ordering and row/target conservation

All 8 explicit checks required for a defensible split, run and printed (not just asserted silently).

In [ ]:
all_pass = validate_temporal_split(df, train_df, val_df, test_df)
assert all_pass, "Temporal split validation FAILED."

## 9. Display class distribution

In [ ]:
summary = summarize_split(train_df, val_df, test_df)
summary

**Note on the uneven fraud rates:** train 0.0816%, validation 0.0575%, test 0.4361%. These are NOT judged against each other -- chronological data is expected to vary naturally over time, and section 4 already explained why: test covers the later, sparser-volume period where fraud counts stay flat while legitimate volume collapses. This is realistic and left as-is rather than corrected for.

## 10. Historical-feature behavior across split boundaries

**The question:** Step 3's historical/velocity features (`prior_sender_*`, `prior_receiver_*`, all velocity features) were computed ONCE over the full 6.36M-row dataset, before this split existed. Does that create leakage now that the data is split by time?

**The reasoning:** every one of those features is computed via `pd.merge_asof(..., direction="backward")` against `step - offset`, which can only ever match rows with a STRICTLY EARLIER step than the current transaction (verified in Step 3, including a same-step controlled test). This means:
- A **train** transaction at step X (X <= 323) can only reference rows with step < X <= 322 -- entirely within the train period. It can never see validation (324-378) or test (379-743) data, no matter how the global computation was run.
- A **validation** transaction at step X (324-378) can reference rows with step < X, which may include train-period AND earlier validation-period rows. This is correct, not leakage: by the time that transaction happens, those earlier validation-period transactions have already genuinely occurred.
- A **test** transaction can reference the entire train + validation history plus earlier test-period rows -- again correct, since a real system would have that whole history available by then.

**We don't just claim this -- we verify it directly** by rebuilding the feature pipeline from raw data restricted to ONLY the train period (steps 1-323) and comparing every resulting feature value against the corresponding rows in the globally-computed dataset.

In [ ]:
raw = load_raw_data()
raw_train_only = raw[raw["step"] <= train_end_step].copy()
print(f"Raw rows with step <= {train_end_step}: {len(raw_train_only):,}")

t0 = time.time()
train_only_features = build_feature_dataset(raw_train_only)
print(f"Rebuilt features from TRAIN-ONLY raw subset in {time.time()-t0:.1f}s -- shape {train_only_features.shape}")

In [ ]:
compare_cols = [c for c in df.columns if c not in ("nameOrig", "nameDest")]
a = train_df.loc[train_only_features.index, compare_cols].sort_index()
b = train_only_features[compare_cols].sort_index()

mismatches = {}
for col in compare_cols:
    if not np.allclose(a[col].values.astype(float), b[col].values.astype(float), equal_nan=True):
        mismatches[col] = int((a[col].values != b[col].values).sum())

if mismatches:
    print("MISMATCHES FOUND (this would indicate a real leakage bug):", mismatches)
else:
    print("RESULT: IDENTICAL across all", len(compare_cols), "compared columns and", f"{len(a):,}", "rows.")
    print("Every train-period feature value is exactly the same whether computed on the full 6.36M-row")
    print("dataset or on the train-only 323-step subset. This empirically confirms -- not just by design,")
    print("but by direct recomputation -- that no train-period feature depends on validation/test data.")

**Conclusion:** it is safe that Step 3 engineered features globally before this split existed. The causal (strictly-earlier-step) design mathematically guarantees a row's features can never reference a later step, so global computation and per-split computation give identical results for every row -- confirmed here by direct recomputation, not just asserted.

## 11. Save split metadata

In [ ]:
print("=== Disk size estimate before saving ===")
for name, split in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    mem_mb = split.memory_usage(deep=True).sum() / 1e6
    print(f"  {name:10s}: {len(split):,} rows, ~{mem_mb:,.1f} MB in memory")

In [ ]:
for name, split in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    out = f"../data/processed/{name}.parquet"
    split.to_parquet(out, index=False, engine="pyarrow", compression="snappy")
    disk_mb = os.path.getsize(out) / 1e6
    print(f"Saved {out}: {disk_mb:,.1f} MB")

In [ ]:
model_cols = get_model_feature_columns(df)

metadata = {
    "dataset_name": "Mobile-Money Fraud Detection using PaySim - engineered feature set",
    "total_rows": int(len(df)),
    "total_fraud": int(df["isFraud"].sum()),
    "train_step_range": [int(df["step"].min()), int(train_end_step)],
    "validation_step_range": [int(train_end_step) + 1, int(val_end_step)],
    "test_step_range": [int(val_end_step) + 1, int(df["step"].max())],
    "train_row_count": int(len(train_df)),
    "validation_row_count": int(len(val_df)),
    "test_row_count": int(len(test_df)),
    "train_fraud_count": int(train_df["isFraud"].sum()),
    "validation_fraud_count": int(val_df["isFraud"].sum()),
    "test_fraud_count": int(test_df["isFraud"].sum()),
    "feature_column_count": len(model_cols),
    "target_column": "isFraud",
    "split_rationale": (
        "Chronological split on `step` (PaySim simulated time), NOT random, because fraud detection is "
        "deployed against future transactions and a random split would let the model train on data that "
        "occurs after some of its own evaluation data. Boundaries (step 323, step 378) were chosen as the "
        "earliest steps at which cumulative ROW COUNT reaches ~70% and ~85% of the dataset respectively, "
        "so each split falls on a complete step boundary. Resulting split is 70.15% / 15.41% / 14.44% by "
        "row count -- not forced to exact thirds because daily transaction volume is highly uneven (it "
        "decays sharply after simulated day ~16). The test period has a substantially higher raw fraud RATE "
        "(0.4361% vs 0.0816% in train) because legitimate volume collapses in later simulated days while "
        "fraud injection stays roughly constant -- a genuine dataset property, documented rather than "
        "corrected for."
    ),
}

with open("../data/processed/split_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved ../data/processed/split_metadata.json")
metadata

## 12. Final summary

- **Boundaries:** train = steps 1-323, validation = steps 324-378, test = steps 379-743.
- **Rows:** train 4,463,587 / validation 980,416 / test 918,617 (70.15% / 15.41% / 14.44%).
- **Fraud counts:** train 3,643 / validation 564 / test 4,006 -- sums to the full dataset's 8,213.
- **All 8 validation checks passed**, including strict step ordering and zero row overlap between splits.
- **Historical-feature leakage check passed by direct recomputation**, not just by design argument: train-period feature values are identical whether computed on the full dataset or a train-only subset.
- **Important limitation:** the test period's much higher fraud rate reflects a real, uncorrected property of the simulated data (legitimate volume collapses late in the simulation while fraud stays roughly constant) -- this will make test-set precision/recall numbers in later steps harder to compare directly against train-period intuition, and should be called out explicitly when reporting Step 7+ results rather than treated as a modeling win or a bug.
- No model was trained in this notebook.